In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
import sys
import os
import inspect
import datetime
import logging

In [ ]:
# filesuffix = f'_{datetime.datetime.now():%y%m%d_%H%M}'
# log_file = os.path.join(scriptdir, 'logs', f'{program_name}_{filesuffix}.log')

logging.basicConfig(
    level=logging.INFO,
    format=('%(asctime)s:%(name)s:%(levelname)s:%(funcName)s:%(message)s'), #  + logging.BASIC_FORMAT
    handlers=[
        # logging.FileHandler(log_file),
        logging.StreamHandler(),
    ]
)
logger = logging.getLogger(__name__)
logging.getLogger('ib_insync.objects').setLevel(logging.WARN)


In [ ]:
import math
import numpy as np
from numpy.typing import NDArray
import pandas as pd
import ipywidgets as widgets
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import pathlib
import inspect

In [ ]:
import more_itertools as mit
from more_itertools import windowed
from itertools import pairwise

In [ ]:
search_dirs = ['../..', '../../..', '..', '../../../jj625/mplfinance/src']
pkg_needed = ['mplfinance', 'eventkit', 'ib_insync']
pkg_dirs = {}

for pkg in pkg_needed:
    for dir in search_dirs:
        potential_dir = pathlib.Path(dir, pkg).resolve()
        if potential_dir.exists() and potential_dir.is_dir():
            pkg_dirs[pkg] = potential_dir
            break

for pkg, pkg_dir in pkg_dirs.items():
    if str(pkg_dir) not in sys.path:
        sys.path.insert(0, str(pkg_dir))
        print(f"{pkg_dir} added to sys.path")

missing_pkgs = [pkg for pkg in pkg_needed if pkg not in pkg_dirs]
if missing_pkgs:
    print(f"Expected directories not found for: {', '.join(missing_pkgs)}")


In [ ]:
# import ib_insync
from ib_insync.objects import BarDataList, BarData

In [ ]:
import mplfinance as mpf
print(f"module loaded: {inspect.getfile(mpf)}")


In [ ]:
# util.logToConsole(logging.INFO)
sym = 'SPY'

date = f'{datetime.datetime.now()+datetime.timedelta(days=0):%Y%m%d}'
filename = f'{sym}_5s_{date}.csv'
bars_df = pd.read_csv(rf'.\data\{filename}', parse_dates=['date'])
if 'open_' in bars_df.columns:
    bars_df.rename(columns={'open_': 'open'}, inplace=True)
bars_df['log_avg'] = np.log(bars_df['average'])

In [ ]:
bars_df.shape

In [ ]:
# Filter bars_df for the time range from 9:30am to 4:00pm
m1 = (bars_df['date'].dt.time >= datetime.time(9, 30)) & (bars_df['date'].dt.time < datetime.time(16, 0))
m2 = (bars_df['date'].dt.time >= datetime.time(9, 30)) & (bars_df['date'].dt.time <= datetime.time(12, 5))
# filtered_bars_df = bars_df[m1]

# Create the date range index
# idx = pd.date_range(start=filtered_bars_df['date'].min(), end=filtered_bars_df['date'].max(), freq='1min')


In [ ]:
logging.getLogger('low_to_high_inflection_point').setLevel(logging.WARN)

In [ ]:
def low_to_high_inflection_point(bars: BarDataList, n=15, m=5) -> bool:
    """check if there is a low to high inflection point in the last n bars.
    This is a sign of reversal. We want to catch the reversal early.
    Define inflection point as any of the last m bars lows is lower than any of the previous n-m bars lows
    """
    if len(bars) < n:
        # the first n bars ...
        # case in point: AMD 10/29/2024 9:35 and 9:36 bars
        if len(bars) > 3:
            lows = [b.low for b in bars] # low so far
            lastn_lows = lows[-3:] # last 3 lows
            if min(lastn_lows) == min(lows):
                pass
                # logger.info(f"low_to_high_inflection_point: last 3 lows are the same: {lows} {lastn_lows}")
                # return True
        return False
    lows = [b.low for b in bars[-n:]]
    nplows = bars.low_prices[:bars._npidx][-n:]
    if not np.isclose(lows, nplows, atol=1e-6).all():
        logger.warning(f"lows != nplows: {lows} != {nplows}")
    if min(lows[:-m]) > min(lows[-m:]):
        # logger.info(f"low_to_high_inflection_point: {lows[:-m]} > {lows[-m:]}")
        return True
    return False # no inflection point


In [ ]:
def low_to_high_inflection_point_new(bars: BarDataList, n=15, m=3) -> bool:
    """check if there is a low to high inflection point in the last n bars.
    an interval contains an inflection point the points before and after it are higher the farther away from the inflection point.
    if plotted, the shape of the interval looks like a 'V' or 'U'
    the eligible inflection points are bars[-m], bars[-m-1], bars[-m-2]
    """
    if len(bars) < n:
        return False

    lows = [b.low for b in bars[-n:]]
    # lows = bars.low_prices[:bars._npidx][-n:]
    eligible_inflection_points = lows[-m-2:-m+1]

    for i in range(len(eligible_inflection_points)):
        if min(lows[:n-m-2+i]) > eligible_inflection_points[i] < min(lows[-m+i+1:]):
            return True

    return False


In [ ]:
# Check for zeros in OHLC columns
ohlc_columns = ['open', 'high', 'low', 'close']
zero_ohlc = bars_df[ohlc_columns].eq(0).any()

if zero_ohlc.any():
	print("Zeros found in OHLC columns:")
	print(zero_ohlc[zero_ohlc].index.tolist())
else:
	print("No zeros found in OHLC columns.")


In [ ]:
with pd.option_context('display.max_columns', 200, 'display.width', 200):
    display(bars_df[m1].head(), bars_df[m1].tail())

In [ ]:
# BarDataList(durationStr='1 D', barSizeSetting='1 min', useRTH=False, from_df=bars_df)

In [ ]:
bars = BarDataList(durationStr='1 D', barSizeSetting='5 secs', useRTH=True, from_df=bars_df[m1])

In [ ]:
len(bars.npdate)

In [ ]:
vlines_10_5 = []
vlines_6_3 = []
vlines_8_3 = []
vlines_15_new = []
df=bars_df[m1].loc[:, ['date', 'open', 'high', 'low', 'close', 'average', 'volume']]

for i in range(1, len(bars)+1):
    # bars = BarDataList(durationStr='1 D', barSizeSetting='5 secs', useRTH=True, from_df=df.iloc[:i])
    bars_subset = bars[:i]
    # if low_to_high_inflection_point(bars, 10, 5):
    #     vlines_10_5.append(df.date.iloc[i-1])
    # if low_to_high_inflection_point(bars, 6, 3):
    #     vlines_6_3.append(df.date.iloc[i-1])
    # if low_to_high_inflection_point(bars, 8, 3):
    #     vlines_8_3.append(df.date.iloc[i-1])
    if low_to_high_inflection_point_new(bars_subset, 15):
        vlines_15_new.append(bars[i-1].date)
        # axs[0].axvline(x=bars[-1].date, color='blue', linestyle='--', linewidth=1)
        # logger.info(f"not inflection point at {bars[-1].date}")

len(vlines_10_5), len(vlines_6_3), len(vlines_8_3), len(vlines_15_new)

In [ ]:
mpf.original_flavor = True
df=bars_df[m1].set_index('date').loc[:, ['open', 'high', 'low', 'close', 'average', 'volume']]
base = df['close'].iloc[0]
df[['open', 'high', 'low', 'close', 'average']] = df[['open', 'high', 'low', 'close', 'average']].apply(lambda x: (x / base) - 1)
df['date'] = df.index

ref = base
def price2pct(px):
    return (px - ref) / ref
def pct2price(pct):
    return pct * ref + ref
fig, axs = mpf.plot(df[:200], style='charles', figratio=(5,1), returnfig=True, vlines=dict(vlines=vlines_15_new[:10], linestyle='--', linewidths=0.8, alpha=0.25, colors='b'))
# axs[0].set_facecolor('#4C4C4C')
secax = axs[0].secondary_yaxis('left', functions=(pct2price, price2pct))
axs[0].yaxis.set_major_locator(ticker.MultipleLocator(0.01))
axs[0].yaxis.set_major_locator(ticker.MultipleLocator(0.006))
axs[0].yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=2))
axs[0].xaxis.set_major_locator(ticker.MultipleLocator(10))

# for i in range(1, len(df)+1):
#     bars = BarDataList(durationStr='1 D', barSizeSetting='1 min', useRTH=True, from_df=bars_df[m1].iloc[:i])
#     if low_to_high_inflection_point(bars, 15, 10):
#         # axs[0].axvline(x=bars[-1].date, color='blue', linestyle='--', linewidth=1)
#         # logger.info(f"not inflection point at {bars[-1].date}")
#         pass
# axs[0].axvline(x=df.date.iloc[10], color='blue', linestyle='--', linewidth=11)
plt.show()